# SIMT 核函数

## 概述

前面两节我们理解了 SIMT 的硬件架构和线程组织方式。本节进入实战：**如何把这套并行模型写成真正的代码？** 也就是如何编写 SIMT 算子的核心——SIMT 核函数（Kernel）。

SIMT 核函数是能在 AI 处理器（NPU）上并行执行、并由主机端（Host）发起调用的函数。它被设计为由大量线程同时并行运行，每个线程拥有独立的寄存器，协同完成海量数据的计算。本节会带你走完一个 SIMT 核函数的完整写法：从定义、调用，到用内置变量定位数据，并提醒一个实际开发中容易忽略的细节。

### 前置要求

- 已理解 Grid、Thread Block、Thread、`gridDim`、`blockDim`、`blockIdx` 和 `threadIdx` 的含义。
- 具备基础 C/C++ 函数定义和指针概念，能够阅读简单函数声明。
- 本小节为理论讲解，示例代码用于理解语法结构，不依赖在线硬件环境执行。

### 学习目标

学完本小节后，你应该能够：

- 写出 SIMT 核函数定义的基本形式，说明 `__global__` 和 `void` 返回类型两条规则。
- 说明 host 侧函数、 SIMT 核函数、device 侧函数之间的调用关系。
- 理解 SIMT 核函数调用符 `<<<...>>>` 中 4 个执行配置参数的含义。
- 说明执行配置参数与 SIMT 核函数内置变量之间的对应关系。
- 读写一个完整的一维 SIMT 核函数，并理解为什么需要边界检查。

### 小节内容

- SIMT 核函数是什么
- SIMT 核函数的定义
- 三类函数的调用关系
- SIMT 核函数的调用与执行配置
- 内置变量与执行配置的对应关系
- 一个完整的向量加法实例
- 一个容易忽略的细节：边界检查

### SIMT 核函数是什么

Ascend C 支持开发者自定义 SIMT 核函数来扩展 C++。和普通 C/C++ 函数最大的不同在于：普通函数在一个线程里顺序执行一次，而**SIMT 核函数一旦启动，就会由成千上万个线程同时并行执行同一份代码**，共同完成数据计算任务。

结合前两节，可以从三个视角理解 SIMT 核函数：

- **硬件视角**：SIMT 核函数运行在 AI 处理器上，线程会使用寄存器、Unified Buffer、Global Memory 等资源。
- **线程视角**：SIMT 核函数启动后，线程按 Grid 和 Thread Block 的层次组织起来并发执行。
- **编程视角**：SIMT 核函数是 host 侧通过特殊调用语法 `<<<...>>>` 启动的 device 侧并行入口。

接下来就从「怎么定义」开始。

### SIMT 核函数的定义

编译器提供特定的**函数类型限定符**来定义 SIMT 核函数，编译器能识别这个函数需要在 Device 侧编译，并允许它从 Host 侧被调用。基本形式如下：

```cpp
// SIMT 核函数定义示例
__global__ void vec_add(float* x, float* y, float* z)
{
    // 具体的计算逻辑
    uint32_t i = blockIdx.x * blockDim.x + threadIdx.x;
    z[i] = x[i] + y[i];
}
```

定义 SIMT 核函数要遵循以下规则：

1. 使用函数类型限定符 `__global__`，标识它是一个 SIMT 核函数。
2. 核函数的返回类型必须是 `void`（计算结果通过出参的方式写到内存中，而不是 return）。

这里要特别理解第 2 条「返回 void」：因为 SIMT 核函数会由成千上万个线程并行执行，没法用单一返回值表达所有线程的结果，所以需要让每个线程把自己算出的结果**写到指针指向的 Global Memory 或共享内存**。上面示例中的 `float* z` 就是这样的输出指针。

### 三类函数的调用关系

一个算子程序中的函数可以分为三类，它们的执行位置和调用关系不同：

<table>
  <thead>
    <tr>
      <th>函数类别</th>
      <th>执行位置</th>
      <th>类型限定符</th>
      <th>调用关系</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>host 侧执行函数</td>
      <td>Host 侧</td>
      <td>无</td>
      <td>可调用其它 host 函数，也可通过 <code>&lt;&lt;&lt;...&gt;&gt;&gt;</code> 调用 SIMT 核函数</td>
    </tr>
    <tr>
      <td>SIMT 核函数</td>
      <td>Device 侧</td>
      <td><code>__global__</code></td>
      <td>可调用除 SIMT 核函数外的其它 device 侧函数</td>
    </tr>
    <tr>
      <td>device 侧执行函数</td>
      <td>Device 侧</td>
      <td><code>__aicore__</code></td>
      <td>可调用同类的其它 device 侧函数</td>
    </tr>
  </tbody>
</table>

下图展示了三类函数之间的调用关系：

![三类函数的调用关系](images/simt/kernel_function_call_relationship.svg)

几个要点：

- Host 侧函数是整个流程的起点，负责准备数据、配置参数，再用 `<<<...>>>` 发起 SIMT 核函数调用。
- SIMT 核函数是 Host 进入 Device 并行计算的**唯一入口**，但 SIMT 核函数**不能再调用另一个 SIMT 核函数**。
- SIMT 核函数内部可以调用用 `__aicore__` 标识的普通 device 侧执行函数，把计算逻辑拆分复用。

### SIMT 核函数的调用与执行配置

Host 侧通过核函数调用符 `<<<...>>>` 来调用 SIMT 核函数。基本形式如下：

```cpp
kernel_name<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>(argument_list);
```

可以把它拆成两部分理解：

- `<<<...>>>`：**执行配置**，告诉硬件用多少线程块、每块多少线程、动态共享内存多大、关联哪个 stream。
- `(argument_list)`：和普通函数一样的参数列表，传给核函数内部使用。

这正是 SIMT 核函数和普通 C/C++ 函数最直观的区别——普通函数调用只写 `func(args)`，核函数调用还必须显式给出**并行执行配置**。

`<<<...>>>` 里的 4 个参数含义如下：

<table>
  <thead>
    <tr>
      <th>参数</th>
      <th>类型</th>
      <th>含义</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>blocks_per_grid</code></td>
      <td>int 或 dim3</td>
      <td>网格的维度与规模，三维乘积 = 线程块总数，不得大于 65535。对应 <code>gridDim</code></td>
    </tr>
    <tr>
      <td><code>threads_per_block</code></td>
      <td>int 或 dim3</td>
      <td>每个线程块的维度与规模，三维乘积 = 每块线程数，不得大于 1024（默认，可配置）。对应 <code>blockDim</code></td>
    </tr>
    <tr>
      <td><code>dyn_ubuf_size</code></td>
      <td>size_t</td>
      <td>为每个线程块动态分配的共享内存字节数，默认 0</td>
    </tr>
    <tr>
      <td><code>stream</code></td>
      <td>aclrtStream</td>
      <td>关联的流，用于维护异步操作的执行顺序</td>
    </tr>
  </tbody>
</table>

初学阶段最该先掌握前两个：`blocks_per_grid` 决定有多少线程块，`threads_per_block` 决定每个线程块多少线程，二者共同决定核函数启动后的并行规模。动态分配共享内存的用法和配置最大线程数的方法将在下一章节详细介绍，此处不展开。

### 内置变量与执行配置的对应关系

核函数启动后，每个线程在函数内可以直接读取**内置变量**，从而知道自己是谁、该处理哪份数据。这些内置变量的取值，正是由前面 `<<<...>>>` 执行配置决定的，两者一一对应：

<table>
  <thead>
    <tr>
      <th>内置变量</th>
      <th>含义</th>
      <th>由哪个执行配置参数决定</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><code>gridDim</code></td>
      <td>网格在各维度上由多少个线程块构成</td>
      <td><code>blocks_per_grid</code></td>
    </tr>
    <tr>
      <td><code>blockDim</code></td>
      <td>每个线程块的线程数</td>
      <td><code>threads_per_block</code></td>
    </tr>
    <tr>
      <td><code>blockIdx</code></td>
      <td>当前线程块在网格中的索引</td>
      <td>运行时由硬件分配</td>
    </tr>
    <tr>
      <td><code>threadIdx</code></td>
      <td>当前线程在所属线程块内的索引</td>
      <td>运行时由硬件分配</td>
    </tr>
  </tbody>
</table>

简单说：`gridDim` 和 `blockDim` 是开发者在调用时**配置进去的规模**，而 `blockIdx` 和 `threadIdx` 是硬件在运行时**分配给每个线程的身份编号**。

至于如何用这几个变量组合出全局数据索引（`blockIdx.x * blockDim.x + threadIdx.x`），在上一章《线程架构》中已有详细讲解，这里不再展开。下面直接看它在核函数里的实际用法。

### 向量加法实例

把前面所有知识点串起来，来看一个完整的核函数是怎么写的。以「向量相加 `z[i] = x[i] + y[i]`」为例，它包含**核函数实现**和 **Host 侧调用**两部分：

```cpp
// 1. 核函数实现
__global__ void vec_add(float* x, float* y, float* z, int vector_length)
{
    // 第一步：计算当前线程对应的全局元素索引
    int work_index = blockIdx.x * blockDim.x + threadIdx.x;

    // 第二步：用全局索引读取数据、计算、写回
    if (work_index < vector_length)
    {
        z[work_index] = x[work_index] + y[work_index];
    }
}

// 2. Host 侧调用示例
int main()
{
    // 假设 Host 和 Device 侧的内存分配与初始化均已完成
    int vector_length = 1000;

    // 每个线程块 256 个线程
    int threads = 256;

    // 按「向上取整」计算需要的线程块数： (length + threads - 1) / threads
    int blocks = (vector_length + threads - 1) / threads; // 结果为 4

    // 启动核函数：4 个线程块、每个线程块 256 个线程
    vec_add<<<blocks, threads, 0, stream>>>(dev_x, dev_y, dev_z, vector_length);

    // ... 后续的 stream 同步和资源释放 ...
    return 0;
}
```

可以看到，一个典型的一维 SIMT 核函数就是这样的结构：

1. **算全局索引**：用内置变量算出 `work_index`，确定当前线程负责哪个元素。
2. **处理数据**：用 `work_index` 从 Global Memory 读入数据、计算、写回结果。

Host 侧则负责配置并行规模（多少线程块、每块多少线程）并发起调用。掌握这套写法，就能写出一个基础的 SIMT 算子了。

### 边界检查

回头看上面核函数里的这行 `if (work_index < vector_length)`，它叫**边界检查**，是实际开发中很容易忽略、却非常重要的一步。

为什么需要它？因为**数据总长度往往无法被总线程数整除**。本例要处理 1000 个元素，按每块 256 线程、向上取整算出 4 个线程块，实际下发了 `4 × 256 = 1024` 个物理线程。当执行到最后一个线程块（`blockIdx.x = 3`）时，末尾 24 个线程算出的 `work_index` 落在 `[1000, 1023]`——已经超出了有效数据范围。

如果没有这道检查，这 24 个线程会去访问 `x[1000]`…`x[1023]` 等不存在的位置，造成**内存越界**。加上 `if (work_index < vector_length)` 后，这些「多余」的线程会直接跳过计算、安全退出，避免了越界风险。

所以一个更完整的核函数写法是「**算全局索引 → 边界检查 → 处理数据**」这三步。在数据长度不确定或不能被整除时，边界检查几乎是必须的。

### 术语速查

<table>
  <thead>
    <tr>
      <th>术语</th>
      <th>说明</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>SIMT 核函数</td>
      <td>在 AI 处理器上并行执行、由 Host 发起调用的 device 侧入口函数。</td>
    </tr>
    <tr>
      <td><code>__global__</code></td>
      <td>标识函数是 SIMT 核函数的类型限定符。</td>
    </tr>
    <tr>
      <td><code>__aicore__</code></td>
      <td>标识除 SIMT 核函数外的 device 侧执行函数。</td>
    </tr>
    <tr>
      <td><code>&lt;&lt;&lt;...&gt;&gt;&gt;</code></td>
      <td>Host 侧调用 SIMT 核函数时使用的执行配置语法。</td>
    </tr>
  </tbody>
</table>

## 运行完整的 SIMT 向量加法

前面拆解了核函数的定义、调用、全局索引和边界检查。下面把它们组装成一个**可以真实编译运行**的完整向量加法算子 `z[i] = x[i] + y[i]`，端到端走一遍「写核函数 → Host 侧申请资源、下发、校验 → 编译 → 运行」。

### 1. 编写核函数与 Host 侧代码

把核函数实现和 Host 侧调用写入同一个 `vec_add.asc`。核函数遵循「**算全局索引 → 边界检查 → 读-算-写**」三步；Host 侧负责申请运行资源、分配 Device 内存、拷入数据、配置并行规模下发、同步、拷回并校验。

In [ ]:
%%writefile vec_add.asc
/**
 * SIMT 向量加法完整示例：z[i] = x[i] + y[i]
 */
#include <cstdio>
#include <vector>
#include "acl/acl.h"
#include "simt_api/asc_simt.h"

// 1. 核函数实现：每个线程处理一个元素
__global__ void vec_add(float* x, float* y, float* z, int vector_length)
{
    int work_index = blockIdx.x * blockDim.x + threadIdx.x;    // 算全局索引
    if (work_index < vector_length) {                          // 边界检查
        z[work_index] = x[work_index] + y[work_index];         // 读-算-写
    }
}

int main()
{
    const int vector_length = 1000;
    const size_t byte_size = vector_length * sizeof(float);

    // Host 侧准备输入数据与期望结果
    std::vector<float> h_x(vector_length), h_y(vector_length), h_z(vector_length, 0.0f);
    for (int i = 0; i < vector_length; ++i) {
        h_x[i] = static_cast<float>(i);
        h_y[i] = static_cast<float>(2 * i);
    }

    // 运行管理资源申请
    aclInit(nullptr);
    aclrtSetDevice(0);
    aclrtStream stream = nullptr;
    aclrtCreateStream(&stream);

    // Device 侧内存分配并拷入输入
    float *d_x = nullptr, *d_y = nullptr, *d_z = nullptr;
    aclrtMalloc((void**)&d_x, byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&d_y, byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMalloc((void**)&d_z, byte_size, ACL_MEM_MALLOC_HUGE_FIRST);
    aclrtMemcpy(d_x, byte_size, h_x.data(), byte_size, ACL_MEMCPY_HOST_TO_DEVICE);
    aclrtMemcpy(d_y, byte_size, h_y.data(), byte_size, ACL_MEMCPY_HOST_TO_DEVICE);

    // 执行配置：每块 256 线程，向上取整算线程块数
    int threads = 256;
    int blocks = (vector_length + threads - 1) / threads;   // = 4
    vec_add<<<blocks, threads, 0, stream>>>(d_x, d_y, d_z, vector_length);
    aclrtSynchronizeStream(stream);

    // 拷回结果并校验
    aclrtMemcpy(h_z.data(), byte_size, d_z, byte_size, ACL_MEMCPY_DEVICE_TO_HOST);
    int errors = 0;
    for (int i = 0; i < vector_length; ++i) {
        if (h_z[i] != h_x[i] + h_y[i]) ++errors;
    }
    printf("vec_add: z[0]=%.1f z[999]=%.1f, %s\n", h_z[0], h_z[999],
           errors == 0 ? "[Success] result verified" : "[Failed] mismatch");

    // 资源释放
    aclrtFree(d_x); aclrtFree(d_y); aclrtFree(d_z);
    aclrtDestroyStream(stream);
    aclrtResetDevice(0);
    aclFinalize();
    return errors == 0 ? 0 : 1;
}

### 2. 命令行编译

本示例的核函数与 Host 侧 `main` 写在同一个 `vec_add.asc` 文件中，直接用毕昇编译器 `bisheng` 一条命令即可编译为可执行文件。编译前先 `source` CANN 的 `set_env.sh` 导入环境变量（让 `bisheng` 及头文件、库路径生效）。

```shell
source ${ASCEND_HOME_PATH}/set_env.sh   # 导入 CANN 环境变量
bisheng vec_add.asc -o vec_add --npu-arch=dav-3510 --enable-simt
```

关键编译选项：

- `--npu-arch=dav-3510`：指定 AI 处理器架构，`dav-3510` 对应 Ascend 950PR/Ascend 950DT。
- `--enable-simt`：指定以 SIMT 方式编译，SIMT 编程场景必须带上。

In [ ]:
!source ${ASCEND_HOME_PATH:-/usr/local/Ascend/cann}/set_env.sh && \
 bisheng vec_add.asc -o vec_add --npu-arch=dav-3510 --enable-simt

### 3. 运行并查看结果

运行编译出的可执行文件。核函数处理 1000 个元素，输入 `x[i]=i`、`y[i]=2i`，因此期望 `z[i]=3i`：`z[0]=0`、`z[999]=2997`，并打印校验结果 `[Success] result verified`。

In [ ]:
!source ${ASCEND_HOME_PATH:-/usr/local/Ascend/cann}/set_env.sh && ./vec_add

## 小节小结

本小节讲解了如何编写 SIMT 算子的核心——SIMT 核函数：

- **定义**：用 `__global__` 标识，返回类型必须是 `void`，计算结果通过指针参数写回 Global Memory，而不是用 return。
- **调用关系**：Host 函数通过 `<<<...>>>` 调用 SIMT 核函数；SIMT 核函数是进入 Device 并行计算的唯一入口，可调用 `__aicore__` 标识的 device 函数，但不能调用另一个 SIMT 核函数。
- **执行配置**：`<<<blocks_per_grid, threads_per_block, dyn_ubuf_size, stream>>>`，前两个参数决定并行规模（对应 `gridDim`、`blockDim`）。
- **边界检查**：数据长度通常无法被线程数整除，需多分配线程并用 `if (work_index < length)` 拦截越界线程。

掌握「定义 → 调用 → 算全局索引 → 边界检查」这条主线，就具备了读写一个基础 SIMT 算子的能力。而要写出**高性能**的算子，还需要理解数据在不同存储介质间如何流动、如何配置——这正是下一章节《内存层级》要讲的内容。

## 课后练习

本节讲解了 SIMT 核函数的定义、调用、内置变量定位数据和边界检查，请根据学习内容完成以下题目进行自测。

1. （判断题）SIMT 核函数的返回类型可以是 `int`，用于把计算结果返回给 Host 侧。

2. （单选题）定义 SIMT 核函数时，必须使用哪个函数类型限定符？  
    A. `__aicore__`  
    B. `__global__`  
    C. `__ubuf__`  
    D. `__device__`  

3. （单选题）要处理 1000 个元素、每块 256 个线程，实际下发 1024 个线程，SIMT 核函数中 `if (work_index < vector_length)` 这行边界检查的作用是？  
    A. 计算线程的全局索引  
    B. 让多余的 24 个越界线程安全退出，防止内存越界  
    C. 提高线程并行度  
    D. 分配动态共享内存  

4. （多选题）以下关于 SIMT 核函数的说法，哪些是正确的？  
    A. Host 侧函数通过 `<<<...>>>` 调用 SIMT 核函数  
    B. SIMT 核函数可以调用用 `__aicore__` 标识的 device 侧函数，但不能调用另一个 SIMT 核函数  
    C. SIMT 核函数可以在内部直接调用另一个 SIMT 核函数，实现多级并行  
    D. SIMT 核函数返回类型必须是 `void`，计算结果通过指针参数写回  

**执行以下代码获取答案。**

## 参考答案

运行下面的单元格查看课后练习参考答案。


In [ ]:
!cat answer/03.04.03_answer.txt
